# Module D.1–D.3: RAG — Why, Chunking, and Retrieval
**Part II — Applied LLM Engineering**

> Ground the model in your documents. Retrieve relevant chunks before you generate.

## 1. Why RAG? The Parametric Memory Problem

A language model stores everything it knows inside its **parameters** — the billions of weights frozen at the end of training. This is called **parametric memory**. It has two hard limits:

1. **Knowledge cutoff.** The model has zero information about events after training ended.
2. **Private knowledge.** The model has never seen your internal documents, your company's data, or anything that was never in the training corpus.

When you ask the model about something outside its parametric memory, it does not say "I don't know." Instead it **confabulates**: it generates a plausible-sounding but fabricated answer. This is the hallucination problem at its worst.

Let's make this concrete with a fictional company no model could have seen.

In [ ]:
# Simulating what a local model would do when asked about unknown facts.
# We don't have the model here, but we can print what hallucinated output looks like.

query = "What were AcmeCorp's Q3 results?"

# This is representative of what a model would confidently generate:
hallucinated_answer = """
AcmeCorp reported Q3 revenue of $2.4 billion, up 12% year-over-year. Net income
came in at $310 million, beating analyst estimates by $0.07 per share. The CEO
cited strong demand in their cloud division as the primary growth driver.
"""

print(f"Query: {query}")
print(f"\nModel output (hallucinated):\n{hallucinated_answer}")
print("Problem: AcmeCorp is fictional. Every number above was invented.")

The model sounded confident. It cited specific numbers. It was completely wrong.

**Retrieval-Augmented Generation (RAG)** fixes this by adding a retrieval step *before* generation. Instead of relying on what the model has memorized, we:

1. **Retrieve** relevant documents from a corpus we control.
2. **Augment** the prompt with those documents as context.
3. **Generate** an answer grounded in the retrieved evidence.

The model's parametric memory is no longer the sole source of truth — it becomes a *reasoning engine* over context we provide.

## 2. The RAG Loop

```
Query → [Retriever] → top-k chunks
                           ↓
                     [Context builder]
                           ↓
Query + Context → [LLM] → Answer
```

**Step 1 — Retrieve**: Given the user query, search an indexed corpus and return the `k` most relevant text chunks.

**Step 2 — Augment**: Concatenate those chunks into a context block. Optionally add source metadata.

**Step 3 — Generate**: Pass `"Context: {chunks}\n\nQuestion: {query}"` to the LLM. The model answers *from the provided context*, not from memory.

In this notebook we build Step 1 from scratch — three different retrieval strategies — because the retriever determines almost everything about RAG quality.

## 3. Our Corpus: The Arborian Codex

We need a corpus that no LLM has ever seen, so that retrieval vs. hallucination is a fair test. We'll use a fictional world — the Arborians, a tree-dwelling civilisation with their own currency, festivals, and capital city.

In [ ]:
CORPUS = """
The Arborians are an ancient people who dwell entirely within the canopy of the
Great Forest, never descending to the forest floor. Their entire civilisation —
homes, markets, temples, and roads — exists among the branches, hundreds of
metres above the ground.

The capital city of the Arborians is called Canopy. It sits at the crown of
the Eldest Oak, a tree so vast that its trunk takes three days to walk around.
Canopy is home to the Council of Leaves, the governing body that sets law for
all Arborian settlements.

Arborian currency is called the Leaflet. One Leaflet is subdivided into one
hundred Veins. Leaflets are minted from compressed birch pulp and bear the
seal of the Council of Leaves on one side and the image of the Eldest Oak on
the other.

Trade between Arborian settlements is conducted via rope-bridge markets called
Strands. Each Strand opens at dawn and closes at dusk. Merchants travel from
branch to branch carrying goods in woven-leaf baskets. The most prized trade
goods are amber resin, sky-silk (woven from spider webs gathered at dawn), and
preserved honey from the cliff-bees of the Northern Canopy.

Arborians celebrate the Festival of Leaves every spring, when the new foliage
returns to the Great Forest. During the festival, each family hangs painted
leaves from their home branch and shares a communal meal called the Green
Table. Children receive small gifts of honey-cake and their first Leaflet coin.

The Festival of Leaves lasts seven days. On the final night, the Council of
Leaves conducts the Ceremony of the Highest Branch, in which the eldest
Arborian citizen climbs to the highest reachable point and speaks the Words of
Renewal to the forest below.

Arborian architecture relies entirely on living wood. Homes are grown, not
built — skilled cultivators called Growers spend years coaxing branches into
the shapes of rooms and staircases. No nails or metal fittings are used; joins
are achieved with bark-lashing and living grafts that seal over time.

The Arborian military is called the Wing Guard. Wing Guards wear cloaks of
compressed moss that blend with the canopy, making them nearly invisible from
below. Their primary weapon is the sling-bow, a compact device that fires
seed-pods with enough force to stun an attacker at thirty metres.

Arborian medicine is practised by specialists called Sap-Healers. They use
preparations of bark extract, distilled dew, and dried fungus. The most
important remedy in the Arborian pharmacopoeia is Greenblood Tincture, made
from the inner bark of the Ironwood tree, which is used to treat the fever
caused by canopy-spider bites.

The Arborian written language, called Leaf-Script, is carved onto polished
bark panels. Books are called Barkfolios. The largest library in Canopy holds
over forty thousand Barkfolios and is tended by a guild of scribes known as
the Bark-Keepers.

Arborian children begin formal education at age five in branch-schools called
Nests. By age twelve, students must pass the Canopy Examination — a test of
climbing, navigation, Leaf-Script reading, and basic arithmetic. Those who pass
with distinction may apply to the Canopy Academy in the capital.

The Arborian calendar divides the year into four seasons named for the forest:
Greening (spring), Fullbranch (summer), Turning (autumn), and Barewood
(winter). Each season has exactly ninety days. The new year begins at the
first dawn of Greening, which coincides with the opening day of the Festival
of Leaves.
"""

print(f"Corpus length: {len(CORPUS)} characters")
print(f"Paragraphs: {len([p for p in CORPUS.strip().split(chr(10)+chr(10)) if p.strip()])}")

## 4. Chunking from Scratch

Before retrieval we need to split the corpus into manageable pieces called **chunks**. The chunk is the unit of retrieval — the smallest piece of text we can return to the LLM as context.

Three classic strategies:

In [ ]:
import re

# --- Strategy A: Fixed-size character chunking with overlap ---

def chunk_fixed(text: str, size: int = 200, overlap: int = 20) -> list[str]:
    """Split text into character-level windows of `size` chars, each overlapping
    the previous chunk by `overlap` characters."""
    chunks = []
    start = 0
    text = text.strip()
    while start < len(text):
        end = start + size
        chunks.append(text[start:end].strip())
        start += size - overlap  # slide forward, keeping `overlap` chars
    return [c for c in chunks if c]  # drop empty trailing chunks


# --- Strategy B: Sentence-based chunking ---

def chunk_sentences(text: str, max_sentences: int = 3) -> list[str]:
    """Split on sentence boundaries (`.` or `\n`), then group into windows of
    `max_sentences` sentences."""
    # Split on period-space or newline; keep the delimiter attached.
    raw_sentences = re.split(r'(?<=[.!?])\s+|\n', text.strip())
    sentences = [s.strip() for s in raw_sentences if s.strip()]
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        group = sentences[i : i + max_sentences]
        chunks.append(' '.join(group))
    return chunks


# --- Strategy C: Paragraph chunking ---

def chunk_paragraphs(text: str) -> list[str]:
    """Split on blank lines (double newline). Each paragraph becomes one chunk."""
    paragraphs = re.split(r'\n\s*\n', text.strip())
    return [p.strip().replace('\n', ' ') for p in paragraphs if p.strip()]


fixed_chunks     = chunk_fixed(CORPUS)
sentence_chunks  = chunk_sentences(CORPUS)
paragraph_chunks = chunk_paragraphs(CORPUS)

print(f"Fixed-size (200 chars, 20 overlap): {len(fixed_chunks):3d} chunks")
print(f"Sentence (3 sentences/group):       {len(sentence_chunks):3d} chunks")
print(f"Paragraph:                          {len(paragraph_chunks):3d} chunks")

In [ ]:
# Inspect one chunk from each strategy to see the difference in granularity.

print("=== Fixed-size chunk #3 ===")
print(repr(fixed_chunks[3]))

print("\n=== Sentence chunk #2 ===")
print(repr(sentence_chunks[2]))

print("\n=== Paragraph chunk #2 ===")
print(repr(paragraph_chunks[2]))

**Chunking tradeoffs:**

| Strategy | Pros | Cons |
|---|---|---|
| **Fixed-size** | Uniform length, easy to tune for token limits | Cuts mid-sentence; overlap adds redundancy but not always the right context |
| **Sentence** | Natural boundaries; complete thoughts | Variable length; short sentences can produce thin chunks |
| **Paragraph** | Highest semantic coherence per chunk | Length is uncontrolled; a long paragraph may exceed the LLM's context window |

For the rest of this notebook we use **paragraph chunks** — each Arborian paragraph is a coherent semantic unit, which makes retrieval results easy to inspect.

In [ ]:
# Use paragraph chunks for all downstream retrieval experiments.
chunks = paragraph_chunks

print(f"Using {len(chunks)} paragraph chunks.")
for i, ch in enumerate(chunks):
    print(f"  [{i:2d}] {ch[:80]}..." if len(ch) > 80 else f"  [{i:2d}] {ch}")

## 5. Dense Retrieval — Vector Search

**Dense retrieval** encodes both queries and documents as dense vectors (embeddings) and retrieves by vector similarity. The intuition: semantically similar text lives nearby in embedding space, even when the words differ.

We use `all-MiniLM-L6-v2` (384 dimensions, outputs L2-normalised vectors so dot product equals cosine similarity).

We implement `VectorIndex` from scratch here — this notebook is self-contained.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

# Load the embedding model once.
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Embedding model loaded. Output dimension: {model.get_sentence_embedding_dimension()}")

In [ ]:
class VectorIndex:
    """Flat brute-force vector index using cosine similarity (dot product on
    normalised vectors). This is what FAISS's IndexFlatIP does under the hood."""

    def __init__(self, embed_fn):
        self._embed = embed_fn   # callable: str | list[str] -> np.ndarray
        self._docs: list[str] = []
        self._matrix: np.ndarray | None = None  # shape (N, D)

    def add(self, docs: list[str]) -> None:
        """Encode and store a list of documents."""
        self._docs = docs
        vecs = self._embed(docs, convert_to_numpy=True, normalize_embeddings=True)
        self._matrix = vecs  # already (N, D)

    def search(self, query: str, top_k: int = 3) -> list[tuple[int, float, str]]:
        """Return top-k (doc_id, score, text) tuples."""
        q_vec = self._embed([query], convert_to_numpy=True, normalize_embeddings=True)[0]
        # Dot product with the whole matrix at once — O(N*D).
        scores = self._matrix @ q_vec          # shape (N,)
        top_ids = np.argsort(scores)[::-1][:top_k]
        return [(int(i), float(scores[i]), self._docs[i]) for i in top_ids]


# Build the index over our paragraph chunks.
vec_index = VectorIndex(embed_fn=model.encode)
vec_index.add(chunks)
print(f"VectorIndex built: {len(chunks)} documents, {vec_index._matrix.shape[1]}-dim embeddings")

In [ ]:
# Dense search: ask about currency using the exact word the corpus uses.
query_currency = "What is the Arborian currency?"
results = vec_index.search(query_currency, top_k=3)

print(f"Query: '{query_currency}'\n")
for rank, (doc_id, score, text) in enumerate(results, 1):
    print(f"  Rank {rank} | doc={doc_id} | score={score:.4f}")
    print(f"  {text[:120]}..." if len(text) > 120 else f"  {text}")
    print()

## 6. Keyword Retrieval — BM25 from Scratch

**BM25** (Best Match 25) is the classic lexical retrieval algorithm. It scores a document based on how often the query terms appear in it, with two important normalisations:

- **IDF (Inverse Document Frequency)**: rare terms get higher weight than common ones.
- **TF saturation**: having 10 occurrences of a word is not 10x better than having 2 — the score saturates.
- **Length normalisation**: long documents are penalised proportionally so they do not dominate by sheer size.

No neural network involved — just counting, ratios, and logarithms.

In [ ]:
import math
from collections import Counter

def tokenize(text: str) -> list[str]:
    """Lowercase word tokeniser — no stopword removal, no stemming."""
    return re.findall(r'\w+', text.lower())


def bm25_score(
    query_tokens: list[str],
    doc_tokens: list[str],
    df: dict[str, int],  # document frequency per term
    N: int,              # total number of documents
    avg_dl: float,       # average document length (in tokens)
    k1: float = 1.5,
    b: float = 0.75,
) -> float:
    """BM25 score for a single document against a query."""
    dl = len(doc_tokens)
    tf_counts = Counter(doc_tokens)
    score = 0.0
    for term in query_tokens:
        if term not in df:
            continue
        tf = tf_counts.get(term, 0)
        # Robertson-Sparck Jones IDF (smoothed to stay positive).
        idf = math.log((N - df[term] + 0.5) / (df[term] + 0.5) + 1)
        # TF with saturation (k1) and length normalisation (b).
        tf_norm = (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avg_dl))
        score += idf * tf_norm
    return score


class BM25Index:
    """In-memory BM25 index. Supports incremental `add` then `search`."""

    def __init__(self):
        self._docs: list[str] = []
        self._tokenised: list[list[str]] = []
        self._df: dict[str, int] = {}    # term -> number of docs containing it

    def add(self, texts: list[str]) -> None:
        """Index a batch of documents."""
        for text in texts:
            tokens = tokenize(text)
            self._docs.append(text)
            self._tokenised.append(tokens)
            # Update document frequency.
            for term in set(tokens):
                self._df[term] = self._df.get(term, 0) + 1

    def search(self, query: str, top_k: int = 3) -> list[tuple[int, float, str]]:
        """Return top-k (doc_id, score, text) tuples."""
        N = len(self._docs)
        avg_dl = sum(len(t) for t in self._tokenised) / max(N, 1)
        q_tokens = tokenize(query)
        scored = [
            (i, bm25_score(q_tokens, self._tokenised[i], self._df, N, avg_dl))
            for i in range(N)
        ]
        scored.sort(key=lambda x: x[1], reverse=True)
        return [(i, s, self._docs[i]) for i, s in scored[:top_k]]


# Build BM25 index.
bm25_index = BM25Index()
bm25_index.add(chunks)
print(f"BM25Index built: {len(chunks)} documents, {len(bm25_index._df)} unique terms")

In [ ]:
# BM25 search on the same currency query.
results_bm25 = bm25_index.search("What is the Arborian currency?", top_k=3)

print("BM25 results — 'What is the Arborian currency?'\n")
for rank, (doc_id, score, text) in enumerate(results_bm25, 1):
    print(f"  Rank {rank} | doc={doc_id} | score={score:.4f}")
    print(f"  {text[:120]}..." if len(text) > 120 else f"  {text}")
    print()

## 7. Comparing Dense vs. Keyword Retrieval

Dense and keyword retrieval have complementary failure modes:

- **Keyword (BM25) wins** on exact-match queries — if the query uses the same rare word the corpus uses, BM25 finds it instantly.
- **Dense wins** on semantic queries — if the query uses synonyms or paraphrases the corpus's phrasing differently, BM25 misses; dense retrieval bridges the vocabulary gap.

Let's run three queries designed to stress each method.

In [ ]:
def show_comparison(query: str, vec_idx: VectorIndex, bm25_idx: BM25Index) -> None:
    vec_results  = vec_idx.search(query, top_k=1)
    bm25_results = bm25_idx.search(query, top_k=1)

    print(f"Query: '{query}'")
    vid, vscore, vtext = vec_results[0]
    bid, bscore, btext = bm25_results[0]
    print(f"  Dense : doc={vid}, score={vscore:.4f} | {vtext[:90].strip()}...")
    print(f"  BM25  : doc={bid}, score={bscore:.4f} | {btext[:90].strip()}...")
    print()

queries = [
    "Festival of Leaves",                    # keyword-friendly: exact phrase in corpus
    "What do Arborians use as money?",       # semantic: 'money' ≠ 'currency' / 'Leaflet'
    "capital city",                          # both should find Canopy; slight differences
]

for q in queries:
    show_comparison(q, vec_index, bm25_index)

In [ ]:
# Deeper look at the semantic gap query.
print("=== Dense top-3 for 'What do Arborians use as money?' ===")
for rank, (doc_id, score, text) in enumerate(vec_index.search("What do Arborians use as money?", top_k=3), 1):
    print(f"  [{rank}] score={score:.4f} | {text[:100]}...")

print()
print("=== BM25 top-3 for 'What do Arborians use as money?' ===")
for rank, (doc_id, score, text) in enumerate(bm25_index.search("What do Arborians use as money?", top_k=3), 1):
    print(f"  [{rank}] score={score:.4f} | {text[:100]}...")

**Observations:**

- "Festival of Leaves" — BM25 scores this high because the exact phrase appears in the corpus. Dense does well too since the phrase is distinctive, but BM25's IDF gives it a large signal boost on the rare exact terms.
- "What do Arborians use as money?" — The corpus uses "currency" and "Leaflet", not "money". BM25 finds no direct match for "money" so it retrieves less relevant chunks. Dense retrieval bridges the lexical gap: "money" and "currency" are close in embedding space.
- "capital city" — both find the Canopy paragraph, but for different reasons: BM25 matches "capital" and "city" literally; dense matches the concept.

## 8. Hybrid Retrieval — Reciprocal Rank Fusion

Since dense and keyword retrieval have complementary strengths, the obvious solution is to **combine them**. But how do you merge two ranked lists with incomparable score scales?

**Reciprocal Rank Fusion (RRF)** solves this elegantly: ignore the raw scores entirely and work only with *ranks*. Each document gets a contribution of `1 / (k + rank)` from each list, where `k` is a smoothing constant that dampens the influence of very high ranks.

In [ ]:
def rrf(rankings: list[list[int]], k: int = 60) -> list[int]:
    """Reciprocal Rank Fusion over multiple ranked lists of document IDs.

    rankings: list of lists, each inner list is a doc-ID ranking (best first).
    k: smoothing constant (default 60, from the original RRF paper).
    Returns: merged ranking, best first.
    """
    scores: dict[int, float] = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)


class HybridIndex:
    """Combines VectorIndex and BM25Index via RRF."""

    def __init__(self, vec_idx: VectorIndex, bm25_idx: BM25Index):
        self._vec = vec_idx
        self._bm25 = bm25_idx

    def search(self, query: str, top_k: int = 3, candidate_k: int = 10) -> list[tuple[int, float, str]]:
        """Retrieve `candidate_k` results from each index, fuse, return top_k."""
        vec_results  = self._vec.search(query, top_k=candidate_k)
        bm25_results = self._bm25.search(query, top_k=candidate_k)

        vec_ranking  = [doc_id for doc_id, _, _ in vec_results]
        bm25_ranking = [doc_id for doc_id, _, _ in bm25_results]

        fused_order = rrf([vec_ranking, bm25_ranking])

        # Look up text from the vector index's document store.
        docs = self._vec._docs
        # Compute a normalised RRF score for display.
        rrf_scores: dict[int, float] = {}
        for ranking in [vec_ranking, bm25_ranking]:
            for rank, doc_id in enumerate(ranking):
                rrf_scores[doc_id] = rrf_scores.get(doc_id, 0.0) + 1.0 / (60 + rank + 1)

        return [(doc_id, rrf_scores[doc_id], docs[doc_id]) for doc_id in fused_order[:top_k]]


hybrid_index = HybridIndex(vec_index, bm25_index)
print("HybridIndex ready (VectorIndex + BM25Index fused with RRF).")

In [ ]:
# Compare all three methods on all three queries side by side.

def top1_text(results):
    _, _, text = results[0]
    return text[:80].strip()

print(f"{'Query':<42} {'Dense':>6}  {'BM25':>6}  {'Hybrid':>6}")
print("-" * 70)

for q in queries:
    vr = vec_index.search(q, top_k=1)
    br = bm25_index.search(q, top_k=1)
    hr = hybrid_index.search(q, top_k=1)

    vid, vscore, _ = vr[0]
    bid, bscore, _ = br[0]
    hid, hscore, _ = hr[0]

    print(f"{q:<42} doc={vid}  doc={bid}  doc={hid}")
    print(f"  Dense  top1: {vr[0][2][:70]}")
    print(f"  BM25   top1: {br[0][2][:70]}")
    print(f"  Hybrid top1: {hr[0][2][:70]}")
    print()

In [ ]:
# Demonstrate RRF rescuing the BM25 failure case.
q = "What do Arborians use as money?"
print(f"Query: '{q}'\n")

print("-- Dense top-3 --")
for doc_id, score, text in vec_index.search(q, top_k=3):
    print(f"  doc={doc_id} score={score:.4f} | {text[:80]}")

print("\n-- BM25 top-3 --")
for doc_id, score, text in bm25_index.search(q, top_k=3):
    print(f"  doc={doc_id} score={score:.4f} | {text[:80]}")

print("\n-- Hybrid (RRF) top-3 --")
for doc_id, score, text in hybrid_index.search(q, top_k=3):
    print(f"  doc={doc_id} rrf_score={score:.6f} | {text[:80]}")

RRF succeeds by letting the dense signal lift the correct currency chunk — even when BM25 ranked it poorly — into the fused top-3. The fused list is more robust than either alone.

## 9. Production Tools Callout

You just built what the following production systems do:

| What we built | Production equivalent |
|---|---|
| `VectorIndex` (brute-force dot product) | FAISS `IndexFlatIP`, Qdrant, Weaviate, Pinecone |
| `BM25Index` | Elasticsearch / OpenSearch BM25, Lucene |
| `HybridIndex` with RRF | LangChain `EnsembleRetriever`, Weaviate hybrid search, Azure AI Search |
| `chunk_paragraphs` / `chunk_fixed` | LangChain `RecursiveCharacterTextSplitter`, LlamaIndex node parsers |

The ideas are identical. The production versions add:

- **Persistence** — indexes survive restarts (stored on disk or in a database).
- **HNSW indexing** — approximate nearest-neighbor graphs that search a million vectors in milliseconds instead of scanning all of them.
- **Horizontal scale** — sharding and replication across machines.
- **Metadata filtering** — "retrieve only chunks from documents uploaded after 2024-01-01" — a SQL-style predicate on document attributes combined with vector search.

Everything else is the math you just coded.

## 10. Try It Yourself

Three tasks to extend what you built:

**Task (a): Chunk metadata.** Add a `metadata` dict to each chunk (paragraph index, character offset in the original text). Surface it in search results so the caller knows *where* a chunk came from.

**Task (b): Sliding-window chunking.** Implement `chunk_sliding(text, window=3, stride=1)` that groups sentences into windows of `window` sentences, advancing by `stride` sentences per step. This creates overlapping sentence-level chunks and is a middle ground between fixed-size and paragraph chunking.

**Task (c): Score threshold in VectorIndex.** Add a `min_score: float = 0.0` parameter to `VectorIndex.search()` that filters out results below that cosine similarity. Try `min_score=0.4` and observe how many chunks survive for different queries — this prevents the retriever from returning irrelevant chunks just because they are the least-bad option.

In [ ]:
# --- Task (a) starter: chunk metadata ---

def chunk_paragraphs_with_metadata(text: str) -> list[dict]:
    """Like chunk_paragraphs but returns dicts with 'text', 'para_idx', 'char_offset'."""
    raw_paragraphs = re.split(r'\n\s*\n', text.strip())
    result = []
    char_offset = 0
    for i, para in enumerate(raw_paragraphs):
        para_clean = para.strip().replace('\n', ' ')
        if not para_clean:
            continue
        result.append({
            'text': para_clean,
            'para_idx': i,
            'char_offset': text.find(para.strip()),  # approximate offset
        })
        char_offset += len(para) + 2  # +2 for the double newline separator
    return result

meta_chunks = chunk_paragraphs_with_metadata(CORPUS)
print(f"Chunks with metadata: {len(meta_chunks)}")
print("Example:")
for field, val in meta_chunks[2].items():
    display_val = val if field != 'text' else val[:80] + '...'
    print(f"  {field}: {display_val}")

In [ ]:
# --- Task (b) starter: sliding-window sentence chunking ---

def chunk_sliding(text: str, window: int = 3, stride: int = 1) -> list[str]:
    """Sentence-level sliding window. `window` = sentences per chunk, `stride` = step."""
    raw = re.split(r'(?<=[.!?])\s+|\n', text.strip())
    sentences = [s.strip() for s in raw if s.strip()]
    chunks = []
    for i in range(0, len(sentences) - window + 1, stride):
        chunks.append(' '.join(sentences[i : i + window]))
    return chunks

sliding_chunks = chunk_sliding(CORPUS, window=3, stride=1)
print(f"Sliding-window chunks (window=3, stride=1): {len(sliding_chunks)}")
print("\nFirst 3 chunks:")
for c in sliding_chunks[:3]:
    print(f"  {c[:100]}...")

In [ ]:
# --- Task (c) starter: min_score threshold in VectorIndex ---

class VectorIndexWithThreshold(VectorIndex):
    """VectorIndex extended with a min_score filter."""

    def search(self, query: str, top_k: int = 3, min_score: float = 0.0):
        """Like VectorIndex.search but drops results below min_score."""
        all_results = super().search(query, top_k=len(self._docs))
        filtered = [(i, s, t) for i, s, t in all_results if s >= min_score]
        return filtered[:top_k]

thresh_index = VectorIndexWithThreshold(embed_fn=model.encode)
thresh_index.add(chunks)

for threshold in [0.0, 0.3, 0.5, 0.7]:
    results = thresh_index.search("What is the Arborian currency?", top_k=5, min_score=threshold)
    print(f"min_score={threshold}: {len(results)} results survive")